<a href="https://colab.research.google.com/github/fcofdezmx/Black_Belt_Project/blob/main/HCO_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HCO assitant with RAG**



## **Problem Definition**

HCO is a Cisco Crosswork solutio.

## **How HCO assitant can help**

We will use a **RAG** model to answer questions from HCO documents.







# **Setup**

In [1]:
# We will use the following lines to filter warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28  --force-reinstall --upgrade --no-cache-dir -q 2>/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 28.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 171.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 174.6 MB/s eta 0:00:00


In [3]:
# For installing the libraries & downloading models from HF Hub
!pip install -q tiktoken==0.6.0 \
                pypdf==4.0.1 \
                langchain==0.1.1 \
                langchain-community==0.0.13 \
                chromadb==0.4.22 \
                sentence-transformers==2.3.1 \
                huggingface_hub==0.23.2 \
                numpy==1.25.2 2>/dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.0/284.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.9 MB/s e

In [4]:
import json
import tiktoken

import pandas as pd

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

from google.colab import userdata, drive

## **LLAMA-CPP**

Add your `HF_TOKEN` in the Secrets section on the left-hand side above the Files menu.
- In the name field enter `HF_TOKEN`.
- In the value field, enter your `access_token`.


In [5]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [6]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf" # the model is in gguf format

In [7]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )


llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [8]:
lcpp_llm = Llama(
        model_path=model_path,
        n_threads=2,  # CPU cores
        n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
        n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
        n_ctx=4096,  # Context window
    )

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [17]:
response = lcpp_llm("Tell me about yourself.", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)

Llama.generate: prefix-match hit



I am a 48-year-old woman, married with two adult children. I have been working as an administrative assistant for the past 20 years. My husband and I are both self-employed and run our own businesses from home. We have a comfortable life but feel that something is missing. We have always been interested in traveling and experiencing new cultures, but our busy schedules and financial responsibilities have made it difficult to do so.

I have a passion for photography and love capturing moments and beauty through my lens. I also enjoy hiking, biking, and spending time outdoors. My family and friends mean everything to me, and I cherish the time we spend together.

Overall, I would say that I am a down-to-earth, practical person who values honesty, integrity, and loyalty. I am excited about the possibility of exploring new opportunities and experiences, but also nervous about stepping out of my comfort zone.


In [18]:
response = lcpp_llm("What is a flower?", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)

Llama.generate: prefix-match hit




A flower is the reproductive structure of a plant, characterized by colorful petals and fragrant scent. Flowers are also known as blooms or blossoms, and they play an important role in the life cycle of plants.

There are many different types of flowers, including:

1. Roses - one of the most popular and romantic flowers, known for their beauty and fragrance.
2. Tulips - a spring flower that comes in a variety of colors and shapes, often associated with love and happiness.
3. Sunflowers - bright yellow flowers with large petals and a tall stature, symbolizing warmth and joy.
4. Daisies - small, delicate flowers with white petals and a yellow center, representing innocence and purity.
5. Lilies - elegant and graceful flowers that come in a range of colors, often associated with refinement and elegance.
6. Orchids - exotic and rare flowers known for their beauty and sophistication, symbolizing luxury and refinement.
7. Carnations - long-lasting flowers that come in a variety of colors,

In [19]:
response = lcpp_llm("How to add and adapter in Cisco Crossworks HCO version 10 ?", max_tokens=500)
response_text = response["choices"][0]["text"]
print(response_text)

Llama.generate: prefix-match hit




Question: How and adapter can be added in Cisco Crosswork HCO version 10?
Answer: In Cisco Crosswork HCO version 10, adapters can be added using the Adapter Designer tool. Here are the steps to add an adapter:

Step 1: Open the Adapter Designer tool by navigating to the "Design" tab and clicking on "Adapter Designer".

Step 2: Click on "New" to create a new adapter.

Step 3: Select the type of adapter you want to create. For example, if you want to create an ODBC adapter, select "ODBC" from the list of available adapters.

Step 4: Provide a name for your adapter and optionally provide a description.

Step 5: In the "Connections" tab, define the connections that your adapter will use. For example, if you are creating an ODBC adapter, you will need to define the database connection details.

Step 6: In the "Operations" tab, define the operations that your adapter will perform. For example, if you are creating an ODBC adapter, you may want to define a "Query" operation to retrieve data 

# **Implementing RAG**

## **1 - Loading the PDF, Chunking**

In [20]:
pdf_file = "/content/Cisco_Crosswork_Hierarchical_Controller_Administration_Guide.pdf"

In [21]:
pdf_loader = PyPDFLoader(pdf_file);

Here we have used the `PyPDFLoader` because we are working with a single document. Suppose we were dealing with multiple documents in various files within a folder. In that case, we would use the `PyPDFDirectorLoader` to point to this folder. It would then load each file, break it into chunks, and store these chunks in a list. This process involves looping over each file in the directory, chunking the file, and storing the chunks.


In [22]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [23]:
microsoft_chunks = pdf_loader.load_and_split(text_splitter)

(Note: Expect that the above cell will take time to execute).

In [24]:
len(microsoft_chunks)

133

## **2 - Vector Store - ChromaDB, Embeddings**

In [25]:
HCO_ADMIN_GUIDE = 'HCO_admin'

In [26]:
import numpy as np
np.__version__

'1.26.4'

In [27]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [28]:
vectorstore = Chroma.from_documents(
    microsoft_chunks,
    embedding_model,
    collection_name=HCO_ADMIN_GUIDE
)

In [29]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

## **3 - RAG Q&A**

### **Prompt Design**

In [30]:
qna_system_message = """
You are an assistant whose work is to review the HCO guide and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [31]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### **Retrieving the Relevant Documents**

In [32]:
user_input = "How and adapter can be added in Cisco Crossworks HCO version 10 ?"

In [33]:
relevant_document_chunks = retriever.get_relevant_documents(user_input)

In [34]:
len(relevant_document_chunks)

5

In [35]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))
    break

Cisco Crosswork Hierarchical Controller 10.0 Administration Guide  
© 2024 Cisco and/or its affiliates . All rights reserved.  Page 64 of 114 To add a device:  
1. In the applications bar in Crosswork Hierarchical Controller, select Services > Device Manager . A list of 
the adapters appears in the Adapters  pane.  
2. Select the Managed Devices  tab. 
3. Click Add Device . 
4. In the General  tab, enter the Name . 
5. In Network Element Site , click to select a site where the device is located.  
 
6. Select the network element  from the list or select the 3D Explorer  tab to s elect the network element on 
the map.  
7. Click OK.


### **Defining the RAG function for response**




In [36]:
def RAG(user_input):
    """
    Args:
    user_input: Takes a user input for which the response should be retrieved from the vectorDB.
    Returns:
    relevant context as per user query.
    """
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)



    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""


    # Quering an LLM
    try:
        response = lcpp_llm(
                prompt=prompt,
                max_tokens=500,
                temperature=0,
                top_p=0.95,
                repeat_penalty=1.2,
                top_k=50,
                stop=['INST'],
                echo=False
                )

        prediction =  response["choices"][0]["text"]

    except Exception as e:
        prediction = f'Sorry, I encountered the following error: \n {e}'

    return  prediction

In [39]:
print(RAG("How to add and adapter in Cisco Crossworks HCO version 10 ?"))

Llama.generate: prefix-match hit


  Sure! I'd be happy to help you with your question based on the provided context.

To add an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the Managed Devices tab.
3. Click Add Device.
4. In the General tab, enter the Name of the adapter.
5. In Network Element Site, click to select the network element in Explorer.
6. Select the Adapters tab and click Assign Device to a new adapter.
7. Select an Adapter and click Assign.

That's it! You have successfully added an adapter in Cisco Crosswork HCO version 10.


In [40]:
print(RAG("How to assign a device to adapter can be added in Cisco Crossworks HCO version 10 ?"))

Llama.generate: prefix-match hit


  Sure, I'd be happy to help! Based on the context you provided, here is the answer to your question:

To assign a device to an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the Managed Devices tab.
3. Click on the required device row (not on the link in the Name column).
4. Select the Adapters tab.
5. Click Assign device to a new adapter.
6. Select an adapter and click Assign.
7. Complete the details for the adapter, such as host, port, direct connect, authentication, and enabled.
8. Repeat these steps for as many adapters as required.
9. Click Add Device to add a device to the list of managed devices.
10. Follow the remaining steps in the guide to complete the assignment of devices to adapters.

I hope this helps! Let me know if you have any other questions or need further clarification.


In [41]:
print(RAG("Can you explain what are Regions API ?"))

Llama.generate: prefix-match hit


  Sure, I can help with that! Based on the context provided, it seems like Regions API is a feature in Cisco Crosswork Hierarchical Controller that allows for managing regions and overlays. A region is defined as a geographical area where network sites are located, and an overlay is used to group several regions together.

The Regions API provides endpoints for querying the model to return the region definition, getting the sites in one or more regions, adding regions to an overlay, and getting the sites in an overlay. The API supports various geometry types for regions, including Point, LineString, Polygon, MultiPoint, MultiLineString, and MultiPolygon.

The context also mentions that Cisco will usually collaborate with customers to set up the regions in their model, and that the regions can be exported or imported in GeoJSON or Region POJOs format.


## **4 - Evaluation**

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Llama model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [42]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [43]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [44]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [45]:
user_input = "How to add and adapter in Cisco Crossworks HCO version 10 ?"

In [46]:
relevant_document_chunks = retriever.get_relevant_documents(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [47]:
# Combine user_prompt and system_message to create the prompt
prompt = f"""[INST]{qna_system_message}\n
            {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
            [/INST]"""

response = lcpp_llm(
        prompt=prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

answer =  response["choices"][0]["text"]

Llama.generate: prefix-match hit


In [48]:
print(answer)

  Sure! I'd be happy to help you with your question based on the provided context.

To add an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, follow these steps:

1. In the applications bar, select Services > Device Manager.
2. Select the Managed Devices tab.
3. Click Add Device.
4. In the General tab, enter the Name of the adapter.
5. In Network Element Site, click to select the network element in Explorer.
6. Select the Adapters tab and click Assign Device to a new adapter.
7. Select an Adapter and click Assign.

That's it! You have successfully added an adapter in Cisco Crosswork HCO version 10.


In [49]:
# Combine user_prompt and system_message to create the prompt
groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
            {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
            [/INST]"""

response = lcpp_llm(
        prompt=groundedness_prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

print(response["choices"][0]["text"])

Llama.generate: prefix-match hit


  Sure, I can help you evaluate the answer based on the provided context and metric. Here are the steps to follow:

Step 1: Evaluate if the answer is derived only from the information presented in the context.

The answer provides a step-by-step guide for adding an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10, which is based on the information provided in the context. Therefore, the metric is followed to a good extent.

Step 2: Evaluate if the answer adheres to the metric considering the question and context as input.

The question asks how to add an adapter in Cisco Crosswork HCO version 10, and the answer provides clear instructions on how to do so. The answer mentions all the necessary steps required to add an adapter, including selecting the Managed Devices tab, clicking Add Device, entering the Name of the adapter, selecting the network element in Explorer, and assigning the device to an adapter. Therefore, the metric is followed completely.

Step 3: Evaluat

In [50]:
# Combine user_prompt and system_message to create the prompt
relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
            {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
            [/INST]"""

response = lcpp_llm(
        prompt=relevance_prompt,
        max_tokens=500,
        temperature=0,
        top_p=0.95,
        repeat_penalty=1.2,
        top_k=50,
        stop=['INST'],
        echo=False
        )

print(response["choices"][0]["text"])

Llama.generate: prefix-match hit


  Sure, I can help you rate the context as per the evaluation criteria based on the given question and answer. Here are the steps to evaluate the context:

Step 1: Identify the main aspects of the question.
The main aspects of the question are:

* Adding an adapter in Cisco Crosswork Hierarchical Controller (HCO) version 10.
* The process involves selecting a network element, assigning the device to an adapter, and configuring the adapter settings.

Step 2: Evaluate how well the answer addresses the main aspects of the question.
The answer provides clear instructions on how to add an adapter in Cisco Crosswork HCO version 10. It covers all the main aspects of the question, including selecting a network element, assigning the device to an adapter, and configuring the adapter settings. The answer is relevant and complete, addressing all the important aspects of the question.

Step 3: Evaluate the extent to which the metric is followed.
The context follows the metric completely as it prov

# **Conclusion**
- We have learned how to create a Retrieval-Augmented Generation (RAG) based application, which can perform Q&A from documents for quicker, more efficient, and accurate information retrieval.